# External Database Setup - CultPass

This notebook sets up the CultPass external database with users, experiences, subscriptions, and reservations.

In [1]:
from datetime import datetime, timedelta
import json
import uuid
import random
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import cultpass

## CultPass Database

### Init DB

In [2]:
cultpass_db = "data/external/cultpass.db"

In [3]:
reset_db(cultpass_db)

Removed existing data/external/cultpass.db
2026-07-02 18:15:06,697 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-07-02 18:15:06,697 INFO sqlalchemy.engine.Engine COMMIT
Recreated data/external/cultpass.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{cultpass_db}", echo=False)
cultpass.Base.metadata.create_all(engine)

### Experiences

In [5]:
experience_data = []

with open('data/external/cultpass_experiences.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        experience_data.append(json.loads(line))

experience_data

[{'title': 'Carnival History Tour in Olinda',
  'description': "Discover the origins and vibrant traditions of Pernambuco's Carnival.",
  'location': 'Pernambuco, Brazil'},
 {'title': 'Sunset Paddleboarding',
  'description': 'Glide across calm waters at golden hour with all gear included.',
  'location': 'Santa Catarina, Brazil'},
 {'title': 'Pelourinho Colonial Walk',
  'description': 'Wander through colorful streets and learn about Afro-Brazilian history.',
  'location': 'Bahia, Brazil'},
 {'title': 'Samba Night at Lapa',
  'description': 'Dance the night away at a traditional samba club in the Lapa arches.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Christ the Redeemer Experience',
  'description': 'Take a guided trip to one of the New Seven Wonders of the World with historical context.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Modern Art at MASP',
  'description': 'Enjoy a guided visit to the São Paulo Museum of Art with insights into its top collections.',

In [6]:
with get_session(engine) as session:
    experiences = []

    for idx, experience in enumerate(experience_data):
        exp = cultpass.Experience(
            experience_id=str(uuid.uuid4())[:6],
            title=experience["title"],
            description=experience["description"],
            location=experience["location"],
            when=datetime.now() + timedelta(days=idx+1),
            slots_available=random.randint(1, 30),
            is_premium=(idx % 2 == 0)
        )
        experiences.append(exp)

    session.add_all(experiences)

### Users

In [7]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

cultpass_users

[{'id': 'a4ab87',
  'name': 'Alice Kingsley',
  'email': 'alice.kingsley@wonderland.com',
  'is_blocked': True},
 {'id': 'f556c0',
  'name': 'Bob Stone',
  'email': 'bob.stone@granite.com',
  'is_blocked': False},
 {'id': '88382b',
  'name': 'Cathy Bloom',
  'email': 'cathy.bloom@florals.org',
  'is_blocked': False},
 {'id': '888fb2',
  'name': 'David Noir',
  'email': 'david.noir@shadowmail.com',
  'is_blocked': True},
 {'id': 'f1f10d',
  'name': 'Eva Green',
  'email': 'eva.green@ecosoul.net',
  'is_blocked': False},
 {'id': 'e6376d',
  'name': 'Frank Ocean',
  'email': 'frank.ocean@seawaves.io',
  'is_blocked': False}]

In [8]:
with get_session(engine) as session:
    db_users = []
    for user_info in cultpass_users:
        user = cultpass.User(
            user_id=user_info["id"],
            full_name=user_info["name"],
            email=user_info["email"],
            is_blocked=user_info["is_blocked"],
            created_at=datetime.now()
        )
        db_users.append(user)
    session.add_all(db_users)

### Subscriptions

In [9]:
with get_session(engine) as session:
    subscriptions = []
    for user_info in cultpass_users:
        subscription = cultpass.Subscription(
            subscription_id=str(uuid.uuid4())[:6],
            user_id=user_info["id"],
            status=random.choice(["active", "cancelled"]),
            tier=random.choice(["basic", "premium"]),
            monthly_quota=random.randint(2, 10),
            started_at=datetime.now()
        )
        subscriptions.append(subscription)

    session.add_all(subscriptions)

### Reservations

In [10]:
with get_session(engine) as session:
    experience_ids = [
        exp.experience_id
        for exp
        in session.query(cultpass.Experience).all()
    ]

    # Reservations for multiple users
    all_reservations = []
    for user_info in cultpass_users[:3]:  # First 3 users get reservations
        for _ in range(2):
            reservation = cultpass.Reservation(
                reservation_id=str(uuid.uuid4())[:6],
                user_id=user_info["id"],
                experience_id=random.choice(experience_ids),
                status="reserved",
            )
            all_reservations.append(reservation)

    session.add_all(all_reservations)
    print(f"Created {len(all_reservations)} reservations")

Created 6 reservations


## Tests

In [11]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    print(f"Users: {len(users)}")
    for user in users:
        print(f"  {user}")

Users: 6
  <User(user_id='a4ab87', email='alice.kingsley@wonderland.com', is_blocked=True)>
  <User(user_id='f556c0', email='bob.stone@granite.com', is_blocked=False)>
  <User(user_id='88382b', email='cathy.bloom@florals.org', is_blocked=False)>
  <User(user_id='888fb2', email='david.noir@shadowmail.com', is_blocked=True)>
  <User(user_id='f1f10d', email='eva.green@ecosoul.net', is_blocked=False)>
  <User(user_id='e6376d', email='frank.ocean@seawaves.io', is_blocked=False)>


In [12]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    for user in users:
        print(f"  {user.full_name}: {user.subscription}")

  Alice Kingsley: <Subscription(subscription_id='22fe91', user_id='a4ab87', status='cancelled', tier='basic')>
  Bob Stone: <Subscription(subscription_id='0f3091', user_id='f556c0', status='active', tier='premium')>
  Cathy Bloom: <Subscription(subscription_id='a8302b', user_id='88382b', status='cancelled', tier='basic')>
  David Noir: <Subscription(subscription_id='32cd45', user_id='888fb2', status='cancelled', tier='premium')>
  Eva Green: <Subscription(subscription_id='1465a6', user_id='f1f10d', status='active', tier='basic')>
  Frank Ocean: <Subscription(subscription_id='a9ed79', user_id='e6376d', status='active', tier='basic')>


In [13]:
with get_session(engine) as session:
    experiences = session.query(cultpass.Experience).all()
    print(f"Experiences: {len(experiences)}")
    for experience in experiences:
        print(f"  {experience}")

Experiences: 7
  <Experience(experience_id='9f7f0e', title='Carnival History Tour in Olinda', when='2026-07-03 18:15:06.718646')>
  <Experience(experience_id='dbf090', title='Sunset Paddleboarding', when='2026-07-04 18:15:06.723097')>
  <Experience(experience_id='03ac11', title='Pelourinho Colonial Walk', when='2026-07-05 18:15:06.723133')>
  <Experience(experience_id='e123a6', title='Samba Night at Lapa', when='2026-07-06 18:15:06.723152')>
  <Experience(experience_id='e1dddf', title='Christ the Redeemer Experience', when='2026-07-07 18:15:06.723169')>
  <Experience(experience_id='1de676', title='Modern Art at MASP', when='2026-07-08 18:15:06.723185')>
  <Experience(experience_id='eeb8e1', title='Ibirapuera Park Bike Ride', when='2026-07-09 18:15:06.723201')>


In [14]:
with get_session(engine) as session:
    reservations = session.query(cultpass.Reservation).all()
    print(f"Reservations: {len(reservations)}")
    for r in reservations:
        print(f"  {r}")

Reservations: 6
  <Reservation(reservation_id='7f8376', user_id='a4ab87', experience_id='03ac11', status='reserved')>
  <Reservation(reservation_id='4f819a', user_id='a4ab87', experience_id='dbf090', status='reserved')>
  <Reservation(reservation_id='cdd868', user_id='f556c0', experience_id='e1dddf', status='reserved')>
  <Reservation(reservation_id='865e9a', user_id='f556c0', experience_id='dbf090', status='reserved')>
  <Reservation(reservation_id='34905b', user_id='88382b', experience_id='1de676', status='reserved')>
  <Reservation(reservation_id='dfa14e', user_id='88382b', experience_id='03ac11', status='reserved')>
